# MediVLM - MIMIC-CXR 1 Epoch Training

This notebook trains the MediVLM model on the full MIMIC-CXR dataset for exactly one epoch.

## 1. Setup Environment

In [ ]:
!git clone https://github.com/sonai-commits/MediVLM.git
%cd MediVLM
!pip install -r requirements.txt
!pip install radgraph RaTEScore
!pip install -e .

Cloning into 'MediVLM'...
remote: Enumerating objects: 88, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (85/85), done.
remote: Total 88 (delta 11), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (88/88), 72.00 KiB | 1.20 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/MediVLM
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=04bcf06f5e78c0647f4f3d7658ea64ff4a8380f069cfe4687b6722365b4965c1
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 588.0/588.0 kB 14.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for radgraph: f

## 2. Patch Dataset to Skip Missing Images

Overwrites the dataset class to filter out missing image paths before training, then reinstalls the package so the new code is picked up.

In [ ]:
%%writefile medivlm/data/mimic_cxr.py
"""MIMIC-CXR dataset (Johnson et al. 2019) - largest CXR-report corpus.

 369K train / 3.0K val / 5.2K test images with
222.8K / 1.8K / 3.3K reports respectively.

Expected annotation format (R2Gen / memory-driven transformer style):
    {
        "train": [
            {"id": "abc123", "image_path": ["path/to/frontal.jpg"],
             "report": "Findings: ..."},
            ...
        ],
        "val":  [...],
        "test": [...]
    }
"""
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path
from typing import Callable, Dict, List, Optional

import torch
from PIL import Image
from torch.utils.data import Dataset

from .transforms import build_image_transform, clean_report


class MimicCxrDataset(Dataset):
    name = "mimic_cxr"

    def __init__(
        self,
        root: str,
        ann_file: str = "annotations.json",
        split: str = "train",
        image_size: int = 224,
        transform: Optional[Callable] = None,
        max_sentences: int = 4,
    ) -> None:
        self.root = Path(root)
        ann_path = self.root / ann_file if not os.path.isabs(ann_file) else Path(ann_file)
        with open(ann_path, "r") as f:
            ann = json.load(f)
        raw_samples: List[Dict] = ann.get(split, [])
        self.split = split
        self.transform = transform or build_image_transform(image_size, train=(split == "train"))
        self.max_sentences = max_sentences
        self.image_dir = self.root / "images"
        if not self.image_dir.exists():
            self.image_dir = self.root

        # Filter out samples whose primary image is missing from disk
        self.samples: List[Dict] = []
        skipped = 0
        for s in raw_samples:
            paths = s.get("image_path", [s.get("image")])
            primary = paths[0] if isinstance(paths, list) else paths
            if (self.image_dir / primary).exists():
                self.samples.append(s)
            else:
                skipped += 1
        if skipped:
            warnings.warn(
                f"MimicCxrDataset [{split}]: skipped {skipped} samples with missing images. "
                f"{len(self.samples)} samples remain."
            )
        print(f"[MimicCxrDataset] {split}: {len(self.samples)} valid samples loaded ({skipped} skipped).")

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int) -> Dict[str, object]:
        s = self.samples[idx]
        paths = s.get("image_path", [s.get("image")])
        if isinstance(paths, list):
            path = paths[0]
        else:
            path = paths
        img = Image.open(self.image_dir / path).convert("RGB")
        image = self.transform(img)
        report = clean_report(s.get("report", ""), max_sentences=self.max_sentences)
        return {
            "id": s.get("id", str(idx)),
            "image": image,
            "report": report,
        }

Overwriting medivlm/data/mimic_cxr.py


In [ ]:
# Reinstall so Python picks up the patched file (clears .pyc cache)
!find . -name '*.pyc' -delete
!pip install -e . -q
print("Patch applied successfully!")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for medivlm (pyproject.toml) ... done
Patch applied successfully!


## 3. Mount Google Drive and Unzip Dataset

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/mimic_cxr.zip'
os.makedirs('data/mimic_cxr', exist_ok=True)

if os.path.exists(ZIP_PATH):
    print(f"\nFound {ZIP_PATH}! Unzipping into data/mimic_cxr...")
    !unzip -q -n "{ZIP_PATH}" -d data/mimic_cxr
    print("Unzip complete!")
else:
    print(f"Error: Could not find {ZIP_PATH}. Please check your Drive path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Found /content/drive/MyDrive/mimic_cxr.zip! Unzipping into data/mimic_cxr...
Unzip complete!


## 4. Generate Annotations from CSV

In [ ]:
import pandas as pd
import json
import ast
import os

print("Reading Kaggle CSVs...")
train_df = pd.read_csv('data/mimic_cxr/mimic_cxr_aug_train.csv')
val_df = pd.read_csv('data/mimic_cxr/mimic_cxr_aug_validate.csv')

def process_df(df):
    samples = []
    for idx, row in df.iterrows():
        try:
            img_list = ast.literal_eval(row['image'])
            corrected_paths = [f"official_data_iccv_final/{p}" for p in img_list]
            samples.append({
                "id": str(row['subject_id']),
                "image_path": corrected_paths,
                "report": str(row['text'])
            })
        except Exception as e:
            continue
    return samples

train_samples = process_df(train_df)
val_samples = process_df(val_df)

annotations = {
    "train": train_samples,
    "val": val_samples,
    "test": val_samples
}

with open('data/mimic_cxr/annotations.json', 'w') as f:
    json.dump(annotations, f)

print(f"\nSaved annotations.json with {len(train_samples)} training samples.")

Reading Kaggle CSVs...

Saved annotations.json with 64586 training samples.


## 5. Create Configuration File (1 Epoch)

In [ ]:
%%writefile configs/mimic_cxr_1_epoch.yaml
detector:
  num_anatomical_classes: 29
  weights_path: null
  top_p_patches: 8
  patch_size: 28
  freeze: true
image_encoder:
  model_name: openai/clip-vit-large-patch14
  image_size: 224
  freeze: true
text_encoder:
  model_name: medicalai/ClinicalBERT
  max_length: 128
  freeze: true
projection:
  proj_dim: 512
  temperature: 0.07
  dropout: 0.1
fusion:
  num_heads: 8
  dropout: 0.1
decoder:
  model_name: gpt2
  trainable_blocks: 4
  max_length: 77
  beam_size: 4
data:
  dataset: mimic_cxr
  root: data/mimic_cxr
  ann_file: annotations.json
  image_size: 224
  batch_size: 32
  num_workers: 2
training:
  epochs: 1
  lr: 2.0e-5
  lambda_ce: 1.0
  lambda_contrast: 0.7
  output_dir: outputs/mimic_cxr_1_epoch


Writing configs/mimic_cxr_1_epoch.yaml


## 6. Run the 1-Epoch Training

In [ ]:
!python scripts/train_supervised.py --config configs/mimic_cxr_1_epoch.yaml

/content/MediVLM/medivlm/data/mimic_cxr.py:67: UserWarning: MimicCxrDataset [train]: skipped 19362 samples with missing images. 45224 samples remain.
  warnings.warn(
[MimicCxrDataset] train: 45224 valid samples loaded (19362 skipped).
/content/MediVLM/medivlm/data/mimic_cxr.py:67: UserWarning: MimicCxrDataset [val]: skipped 162 samples with missing images. 338 samples remain.
  warnings.warn(
[MimicCxrDataset] val: 338 valid samples loaded (162 skipped).
Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100% 83.3M/83.3M [00:00<00:00, 195MB/s]
config.json: 4.52kB [00:00, 12.1MB/s]
model.safetensors: 100% 1.71G/1.71G [00:15<00:00, 113MB/s] 
Loading weights: 100% 391/391 [00:00<00:00, 748.29it/s, Materializing param=vision_model.pre_layrnorm.weight]
CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
---------------

In [ ]:
import os
import sys
from pathlib import Path

# 1. Automatically change directory to the repository in Google Colab
if 'google.colab' in sys.modules or os.path.exists('/content/MediVLM'):
    os.chdir('/content/MediVLM')
    print("Detected Google Colab environment. Changed working directory to /content/MediVLM")

import torch
from PIL import Image
from medivlm.data.transforms import build_image_transform
from medivlm.models import MediVLM
from medivlm.utils import load_checkpoint, load_config

# 2. Define correct paths matching your training outputs
CONFIG_PATH = "configs/mimic_cxr_1_epoch.yaml"
CHECKPOINT_PATH = "outputs/mimic_cxr_1_epoch/best.ckpt"

# Check where 1.jpg is located and resolve its path
if os.path.exists('/content/1.jpg'):
    IMAGE_PATH = '/content/1.jpg'
else:
    IMAGE_PATH = '1.jpg'
print(f"Using image path: {IMAGE_PATH}")

# 3. Load Config and Device Setup
cfg = load_config(CONFIG_PATH)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# 4. Load Model and Weights
model = MediVLM(cfg).to(device)
load_checkpoint(CHECKPOINT_PATH, model=model, map_location=device)
model.eval()
print("Model loaded successfully!")

# 5. Preprocess Image
transform = build_image_transform(cfg.image_encoder.image_size, train=False)
img = Image.open(IMAGE_PATH).convert("RGB")
image_tensor = transform(img).unsqueeze(0).to(device) # Shape: (1, C, H, W)
print(f"Image successfully loaded & transformed to shape: {tuple(image_tensor.shape)}")

# 6. Run Generation
with torch.no_grad():
    reports = model.generate(image_tensor)

print("\n" + "=" * 60)
print(f"[Generated Report for {IMAGE_PATH}]:\n")
print(reports[0])
print("=" * 60)


Detected Google Colab environment. Changed working directory to /content/MediVLM
Using image path: /content/1.jpg
Using device: cuda


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.self_attn.q_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.weight        | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bias              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.v_proj.bias

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: medicalai/ClinicalBERT
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_projector.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully!
Image successfully loaded & transformed to shape: (1, 3, 224, 224)

[Generated Report for /content/1.jpg]:

['findings: impression: in comparison with the study of xxxx, there is little change. no evidence of acute cardiopulmonary disease. no pneumonia, pleural effusion, or pneumothorax.', 'findings': impression: compared to the previous radiograph, the patient is status post median sternotomy. the heart size is normal. the mediastinal
